# 01 - Thu thập dữ liệu repository GitHub

Notebook này gọi module `github_collector.py` để:
1. Tìm repository công khai qua GitHub Search API.
2. Thu thập tối đa 3000 repository **mới** (chưa có trong `data/repositories.csv`).
3. Lấy README, loại trùng theo `github_repo_id`, xuất CSV.
4. Kiểm tra số dòng, giá trị thiếu và tính hợp lệ.

Trước khi chạy, đặt token rồi mới khởi động Jupyter: `export GITHUB_TOKEN=ghp_xxx`

In [6]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import github_collector as gc

print('GITHUB_TOKEN:', 'đã cấu hình' if os.getenv('GITHUB_TOKEN') else 'CHƯA CÓ (rate limit rất thấp)')
print('Số truy vấn (partition):', len(gc.build_partitions()))
print('Số dòng hiện có trong repositories.csv:', len(gc.load_existing_ids()))

GITHUB_TOKEN: đã cấu hình
Số truy vấn (partition): 900
Số dòng hiện có trong repositories.csv: 9000


## 1. Thu thập 3000 repository mới

Lần chạy đầu mất khoảng 10 đến 20 phút vì phải tải README cho từng repository. Các lần sau tiếp tục từ partition kế tiếp nhờ `data/state.json`.

In [7]:
master, checks, missing = gc.run(target=3000, with_readme=True, readme_chars=20000)

[partition 16] language:"Java" stars:50..199 created:2015-01-01..2015-12-31 fork:false | mới: 0/3000


[partition 17] language:"Java" stars:200..999 created:2015-01-01..2015-12-31 fork:false | mới: 1000/3000
[partition 18] language:"Java" stars:1000..4999 created:2015-01-01..2015-12-31 fork:false | mới: 2000/3000
[partition 19] language:"Java" stars:>=5000 created:2015-01-01..2015-12-31 fork:false | mới: 2477/3000
[partition 20] language:"Go" stars:10..49 created:2015-01-01..2015-12-31 fork:false | mới: 2559/3000
Thu thập được 3000 repository mới trong lần chạy này.


## 2. Xem trước dữ liệu

In [8]:
print('Kích thước:', master.shape)
master.head(3)

Kích thước: (12000, 20)


,github_repo_id,full_name,owner_login,name,description,language,topics,readme_text,stars,forks,watchers,open_issues,license,is_fork,is_archived,repo_created_at,repo_updated_at,pushed_at,html_url,collected_at
0,38626348,google/dotty,google,dotty,NaN,Python,[],# EFILTER Query Language\n\nEFILTER is a gener...,49,12,49,7,Apache-2.0,False,True,2015-07-06T15:02:49Z,2026-07-21T08:36:16Z,2021-03-23T15:41:30Z,https://github.com/google/dotty,2026-10-12T04:39:33+00:00
1,47172878,shinyamagami/starting-out-with-python-3rd,shinyamagami,starting-out-with-python-3rd,These are my answers for exercises in Starting...,Python,[],# starting-out-with-python-3rd\nThese codes ar...,49,57,49,0,NaN,False,False,2015-12-01T07:23:52Z,2026-04-12T05:45:31Z,2016-04-04T04:20:17Z,https://github.com/shinyamagami/starting-out-w...,2026-10-12T04:39:33+00:00
2,36799649,BorjaMerino/tlsInjector,BorjaMerino,tlsInjector,Python script to inject and run shellcodes thr...,Python,[],NaN,49,20,49,0,NaN,False,False,2015-06-03T11:36:25Z,2025-05-07T13:18:16Z,2015-06-22T09:15:33Z,https://github.com/BorjaMerino/tlsInjector,2026-10-12T04:39:33+00:00


## 3. Kiểm tra chất lượng dữ liệu

In [9]:
print('Kiểm tra tính hợp lệ:')
for k, v in checks.items():
    print(f'  {k:<35} {v}')

print('\nGiá trị thiếu theo cột:')
print(missing.to_string())

Kiểm tra tính hợp lệ:
  so_dong                             12000
  github_repo_id_trung                0
  full_name_khong_khop_owner_name     0
  gia_tri_dem_am                      0
  html_url_sai_dinh_dang              0
  ngay_khong_parse_duoc               0
  created_sau_updated                 0

Giá trị thiếu theo cột:
github_repo_id        0
full_name             0
owner_login           0
name                  0
description         431
language              0
topics                0
readme_text         110
stars                 0
forks                 0
watchers              0
open_issues           0
license            4026
is_fork               0
is_archived           0
repo_created_at       0
repo_updated_at       0
pushed_at             0
html_url              0
collected_at          0


In [ ]:
print(f'{len(master):,} / 15,000 dòng ({len(master) / 15000:.1%})')
print(sorted(p.name for p in gc.DAILY_DIR.glob('*.csv')))

12,000 / 12,000 dòng (100.0%)
['repositories_20261012.csv', 'repositories_20261014.csv']
